<a href="https://colab.research.google.com/github/rirfan3689/CSCI323-Spam-Detection/blob/main/notebooks/09_spamassassin_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()  # Upload spam_assassin.csv

df = pd.read_csv('spam_assassin.csv')
df.columns = ['message', 'label']
df['label'] = df['label'].map({0: 'ham', 1: 'spam'})
df = df.drop_duplicates()
print("Dataset loaded:", df.shape)

Saving spam_assassin.csv to spam_assassin.csv
Dataset loaded: (5329, 2)


In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import numpy as np

print("Libraries loaded!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Libraries loaded!


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and stem
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['cleaned_message'] = df['message'].apply(preprocess_text)
print("Sample original:", df['message'][0][:100])
print("Sample cleaned:", df['cleaned_message'][0][:100])

Sample original: From ilug-admin@linux.ie Mon Jul 29 11:28:02 2002 Return-Path: <ilug-admin@linux.ie> Delivered-To: y
Sample cleaned: ilugadminlinuxi mon jul returnpath ilugadminlinuxi deliveredto yyyylocalhostnetnoteinccom receiv loc


In [4]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned_message'])
y = df['label']

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

Feature matrix shape: (5329, 5000)
Labels shape: (5329,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nTraining label distribution:")
print(pd.Series(y_train).value_counts())
print("\nTest label distribution:")
print(pd.Series(y_test).value_counts())

Training set size: (4263, 5000)
Test set size: (1066, 5000)

Training label distribution:
label
ham     2910
spam    1353
Name: count, dtype: int64

Test label distribution:
label
ham     728
spam    338
Name: count, dtype: int64


In [6]:
from scipy.sparse import save_npz

save_npz('sa_X_train.npz', X_train)
save_npz('sa_X_test.npz', X_test)
np.save('sa_y_train.npy', y_train)
np.save('sa_y_test.npy', y_test)

print("Processed data saved successfully!")

Processed data saved successfully!
